# Crumblr — Compositional Differential Abundance

Uses **crumblr** (CLR + dream mixed-effects) as an alternative to scCODA.  
Same dataset, same tissue × annotation-level grid.

**Setup (one-time):** `bash /home/gdallagl/myworkdir/XDP/data/crumblr/crumblr/setup_env.sh`  
**Kernel:** any env with `scanpy` + `pandas` (e.g. `python3`/`.venv`) — crumblr runs in its own venv via subprocess.

In [17]:
%load_ext autoreload
%autoreload 2

import os, subprocess, json, glob
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display, Image

# ── Paths ─────────────────────────────────────────────────────────────────────
ADATA          = "/home/gdallagl/myworkdir/XDP/data/XDP/NucSec/original_data/NucSeq_all_QCed_labelled_zoned.h5ad"
SAVE_FOLDER    = "/home/gdallagl/myworkdir/XDP/data/XDP/NucSec/compositional/crumblr"
CRUMBLR_SCRIPT = "/home/gdallagl/myworkdir/XDP/data/crumblr/crumblr/run_workflow.py"
MAPPING_JSON   = "/home/gdallagl/myworkdir/XDP/utils/STR_cell_types_annotation/group_level_mmc_mapping.json"

# ── Study design ──────────────────────────────────────────────────────────────
SAMPLE_COL     = "donor_id"
CONTRAST_COL   = "condition"
CONTRAST_REF   = "Control"
CONTRAST_STIM  = "XDP"
FIXED_EFFECTS  = ["sex", "cohort", "age_of_death"]  # fixed covariates
RANDOM_EFFECTS = []  # donor_id can't be a random effect: 1 sample per donor after aggregation → use lmFit

os.makedirs(SAVE_FOLDER, exist_ok=True)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [18]:
adata = sc.read_h5ad(ADATA, backed="r")
print(adata)
print("Tissues:", adata.obs["tissue"].unique().tolist())

AnnData object with n_obs × n_vars = 502456 × 38601 backed at '/home/gdallagl/myworkdir/XDP/data/XDP/NucSec/original_data/NucSeq_all_QCed_labelled_zoned.h5ad'
    obs: 'background_fraction', 'cell_probability', 'cell_size', 'droplet_efficiency', 'barcode', 'bcl', 'rna_index', 'library', 'library__barcode', 'frac_mito', 'mol_info_nUMI', 'mol_info_nRead', 'frac_intronic', 'donor_id', 'vireo_prob_max', 'vireo_prob_doublet', 'vireo_n_vars', 'vireo_best_singlet', 'vireo_best_doublet', 'vireo_doublet_logLikRatio', 'dropsift_frac_contamination', 'dropsift_training_label_is_cell', 'dropsift_empty_gene_module_score', 'dropsift_is_cell', 'dropsift_is_cell_prob', 'cell_class', 'leiden_0.1', 'leiden_0.2', 'leiden_0.3', 'leiden_0.4', 'leiden_0.5', 'tissue', 'broad_original_cell_type', 'Neighborhood_label', 'Neighborhood_name', 'Neighborhood_bootstrapping_probability', 'Neighborhood_aggregate_probability', 'Neighborhood_correlation_coefficient', 'Class_label', 'Class_name', 'Class_bootstrapping_prob

## Helper functions

In [29]:
import sys
sys.path.insert(0, "/home/gdallagl/myworkdir/XDP/data/crumblr")  # directory, not file path
from plot_crumblr import plot_all as _plot_all

# Columns that every CSV needs (cell-type col is added separately per run)
META_COLS = [SAMPLE_COL, CONTRAST_COL, *FIXED_EFFECTS]

# Subprocess env: override the Jupyter inline backend (invalid outside Jupyter)
_SUBPROCESS_ENV = {**os.environ, "MPLBACKEND": "Agg"}


def drop_bad_samples(adata):
    """Remove donors with any missing covariate."""
    bad = {s for col in META_COLS
              for s in adata.obs[adata.obs[col].isna()][SAMPLE_COL].unique()}
    return adata[~adata.obs[SAMPLE_COL].isin(bad)]


def run_crumblr(adata_sub, ct_col, output_dir, label="", show=False):
    """
    Export a cell-level CSV, call run_workflow.py, then plot results with plot_crumblr.py.
    All CSVs + original PDFs/PNGs are saved to output_dir by run_workflow.py.
    Custom plots saved to output_dir/images/ by plot_crumblr.py.
    Returns the results DataFrame (one row per cell type × coefficient).
    """
    os.makedirs(output_dir, exist_ok=True)
    csv_path = os.path.join(output_dir, "cells.csv")

    # Write cell-level CSV (one row per cell, columns = ct + metadata)
    df = adata_sub.obs[[ct_col, *META_COLS]].dropna().copy()
    df[ct_col] = df[ct_col].astype(str)
    df.to_csv(csv_path, index=False)

    cmd = [
        "python", CRUMBLR_SCRIPT,
        "--csv",            csv_path,
        "--proportion-col", ct_col,
        "--sample-col",     SAMPLE_COL,   # one sample per donor → same as donor-col
        "--donor-col",      SAMPLE_COL,
        "--contrast",       CONTRAST_COL,
        "--ref-level",      CONTRAST_REF,
        "--output-dir",     output_dir,
        "--fixed-effects",  *FIXED_EFFECTS,
        "--random-effects", *RANDOM_EFFECTS,
    ]
    result = subprocess.run(cmd, capture_output=True, text=True, env=_SUBPROCESS_ENV)
    print(result.stdout[-2000:])  # trim output to avoid flooding the notebook
    if result.returncode != 0:
        print("STDERR:", result.stderr[-2000:])
        raise RuntimeError(f"crumblr failed → {output_dir}")

    # Custom plots: use actual cell type names, no internal class mapping
    _plot_all(output_dir, contrast_col=CONTRAST_COL, contrast_stim=CONTRAST_STIM, label=label, show=show)

    return pd.read_csv(os.path.join(output_dir, "crumblr_results.csv"))


def run_crumblr_grid(adata, subfolder, annotation_levels, show=False):
    """Run crumblr for every (tissue, annotation_level) pair."""
    adata = drop_bad_samples(adata).to_memory().copy()

    df_list = []
    for tissue in adata.obs["tissue"].unique():
        adata_t = adata[adata.obs["tissue"] == tissue].copy()
        for ct_col in annotation_levels:
            label = f"{subfolder} | {tissue} | {ct_col}"
            print(f"\n{'='*60}\n{label}")
            print(adata_t.obs[ct_col].value_counts())

            out_dir = f"{SAVE_FOLDER}/{subfolder}/{tissue}/{ct_col}"
            df = run_crumblr(adata_t, ct_col, out_dir, label=label)
            df["tissue"]     = tissue
            df["annotation"] = ct_col
            df_list.append(df)

    merged = pd.concat(df_list, ignore_index=True)
    merged.to_csv(f"{SAVE_FOLDER}/{subfolder}/merged_results.csv", index=False)
    return merged

## All cells

Runs across all tissues for each annotation granularity level.

In [20]:
# spn_type / spn_receptor_type are not obs columns — derive them from Group_name via the JSON mapping
with open(MAPPING_JSON) as f:
    classifications = json.load(f)

for col in ["spn_type", "spn_receptor_type"]:
    if col not in adata.obs.columns:
        adata.obs[col] = adata.obs["Group_name"].map(classifications[col]).fillna(adata.obs["Group_name"])

In [23]:
LEVELS_ALL = ["Class_name", "Subclass_name", "Group_name", "spn_type", "spn_receptor_type"]

df_all = run_crumblr_grid(adata, subfolder="all_cells", annotation_levels=LEVELS_ALL, show=False)
df_all.head()


all_cells | DFC | Class_name
Class_name
OPC-Oligo       42314
CN LGE GABA     39041
CN MGE GABA     19072
Astro-Epen      15292
CN CGE GABA     11187
F M GABA         8323
Immune           4877
Vascular          812
F M Glut          491
Cx GABA           344
CN GABA-Glut      150
M Dopa             94
Name: count, dtype: int64
[1/6] Loading /home/gdallagl/myworkdir/XDP/data/XDP/NucSec/compositional/crumblr/all_cells/DFC/Class_name/cells.csv …
[2/6] Aggregating cell counts …
  Contrast 'condition' detected as categorical.
  Samples: 44  |  Cell types: 11
[3/6] Formula: ~ condition + sex + cohort + age_of_death
[4/6] Running crumblr (R) …

[R] /home/gdallagl/myworkdir/XDP/data/crumblr/micromamba_root/envs/crumblr/bin/Rscript … formula='~ condition + sex + cohort + age_of_death'
── Loading data ────────────────────────────────────────────────────────
  Samples: 44  |  Cell types: 11
  Formula: ~ condition + sex + cohort + age_of_death
  Contrast variable: condition
── Running crumblr ──

,logFC,AveExpr,t,P.Value,adj.P.Val,B,cell_type,coefficient,tissue,annotation
0,0.090436,1.351958,0.724981,0.472448,0.649616,-4.771102,Astro-Epen,conditionXDP,DFC,Class_name
1,0.174112,1.101358,1.551277,0.128272,0.410188,-4.370123,CN CGE GABA,conditionXDP,DFC,Class_name
2,-0.307975,-3.324525,-1.823414,0.075299,0.410188,-4.131381,CN GABA-Glut,conditionXDP,DFC,Class_name
3,0.120077,2.343657,1.342912,0.186449,0.410188,-4.492086,CN LGE GABA,conditionXDP,DFC,Class_name
4,0.044766,1.646579,0.619586,0.538847,0.658591,-4.806643,CN MGE GABA,conditionXDP,DFC,Class_name


## Neurons only

Exclude non-neuronal cells. Groups not in the JSON mapping are labelled `NON_MSN` (the reference category).

In [24]:
NON_MSN = "NON_MSN"  # reference / fill label for non-SPN neurons

adata_n = adata[adata.obs["Neighborhood_name"] != "Nonneuron"].to_memory().copy()

# Add neuron-specific annotation columns (unmapped groups → NON_MSN)
for col in ["Original_group_name", "spn_type", "spn_receptor_type"]:
    if col not in adata_n.obs.columns and col in classifications:
        adata_n.obs[col] = adata_n.obs["Group_name"].map(classifications[col]).fillna(NON_MSN)

LEVELS_NEURONS = ["Original_group_name", "spn_type", "spn_receptor_type"]

df_neurons = run_crumblr_grid(adata_n, subfolder="only_neurons", annotation_levels=LEVELS_NEURONS, show=False)
df_neurons.head()


only_neurons | DFC | Original_group_name
Original_group_name
NON_MSN                        41474
STRv D1 MSN                    17591
STRv D2 MSN                    12291
STRv D1 NUDAP MSN               5256
STRd D2 StrioMat Hybrid MSN      739
STRd D2 Striosome MSN            495
STRd D1 Striosome MSN            393
STR D1D2 Hybrid MSN              224
STRd D2 Matrix MSN               134
STRd D1 Matrix MSN               105
Name: count, dtype: int64
[1/6] Loading /home/gdallagl/myworkdir/XDP/data/XDP/NucSec/compositional/crumblr/only_neurons/DFC/Original_group_name/cells.csv …
[2/6] Aggregating cell counts …
  Contrast 'condition' detected as categorical.
  Samples: 44  |  Cell types: 8
[3/6] Formula: ~ condition + sex + cohort + age_of_death
[4/6] Running crumblr (R) …

[R] /home/gdallagl/myworkdir/XDP/data/crumblr/micromamba_root/envs/crumblr/bin/Rscript … formula='~ condition + sex + cohort + age_of_death'
── Loading data ────────────────────────────────────────────────────────


,logFC,AveExpr,t,P.Value,adj.P.Val,B,cell_type,coefficient,tissue,annotation
0,-0.012158,2.891482,-0.171053,0.864925,0.887391,-4.925152,NON_MSN,conditionXDP,DFC,Original_group_name
1,0.049928,-2.442371,0.360549,0.720070,0.887391,-4.854910,STR D1D2 Hybrid MSN,conditionXDP,DFC,Original_group_name
2,0.025845,-1.907382,0.192893,0.847883,0.887391,-4.957193,STRd D1 Striosome MSN,conditionXDP,DFC,Original_group_name
3,-0.095189,-1.215374,-0.695785,0.490027,0.887391,-4.779011,STRd D2 StrioMat Hybrid MSN,conditionXDP,DFC,Original_group_name
4,-0.213106,-1.683646,-1.790506,0.079886,0.329858,-4.159923,STRd D2 Striosome MSN,conditionXDP,DFC,Original_group_name


## Zones — Striatum - D1 Matrix MSN only

Spatial zones are treated as the cell-type dimension (same as the last scCODA analysis).

Uses curated cell type annotation.

In [ ]:
adata_z = sc.read_h5ad(
    "/home/gdallagl/myworkdir/XDP/data/XDP/NucSec/DEG/Striatum/NucSeq_Striatum_curated_labelels_for_deg_zoned.h5ad",
    backed="r"
)

adata_d1 = adata_z[
    adata_z.obs["Group_name"].isin(["STRd D1 Matrix MSN", "STRv D1 MSN"]) &
    (adata_z.obs["ct_for_deg"] == "Matrix")
].to_memory().copy()

adata_d1 = drop_bad_samples(adata_d1).copy()

# Prefix zone numbers with "zone_" so pandas never parses them as integers
# (run_workflow.py reads counts_matrix.csv and crashes if column names are int)
adata_d1.obs["zone"] = "zone_" + adata_d1.obs["zone"].astype(str)
print(adata_d1.obs["zone"].value_counts())

out_dir_z = f"{SAVE_FOLDER}/zones/Striatum/zone"
df_zones  = run_crumblr(adata_d1, "zone", out_dir_z, label="zones | Striatum | zone", show=False)
df_zones["tissue"]     = "Striatum"
df_zones["annotation"] = "zone"
df_zones.to_csv(f"{SAVE_FOLDER}/zones/results.csv", index=False)
df_zones

## Summary — significant hits across all analyses

In [ ]:
from plot_crumblr import plot_logfc_bar

# Tag and merge everything
df_all["cell_used"]     = "all_cells"
df_neurons["cell_used"] = "only_neurons"
df_zones["cell_used"]   = "zones"

df_merged = pd.concat([df_all, df_neurons, df_zones], ignore_index=True)
df_merged.to_csv(f"{SAVE_FOLDER}/all_results.csv", index=False)

# Re-plot logFC bars per run from saved output directories (reads crumblr_results.csv directly)
for (cell_used, tissue, ann) in df_merged[["cell_used", "tissue", "annotation"]].drop_duplicates().values:
    out_dir = f"{SAVE_FOLDER}/{cell_used}/{tissue}/{ann}"
    print(f"\n{cell_used} | {tissue} | {ann}")
    plot_logfc_bar(out_dir, contrast_stim=CONTRAST_STIM, save=False)